In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 13:56:32 WARN Utils: Your hostname, DESKTOP-JCM8NP2, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 13:56:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/24 13:56:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/24 13:56:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from pathlib import Path

paths = [str(p) for p in Path("data/pq/green/2021").iterdir() if p.is_dir()]

df_green = spark.read.parquet(*paths)

In [3]:
df_green.createOrReplaceTempView('green')

In [9]:
df_green_revenue = spark.sql("""
SELECT 
 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records

FROM
    green
WHERE
    lpep_pickup_datetime >= '2021-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [12]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2021-01-01 00:00:00|   7|             61.47|             2|
|2021-01-01 00:00:00|  17|            102.34|             3|
|2021-01-01 00:00:00|  35|              50.3|             2|
|2021-01-01 00:00:00|  39|              36.0|             1|
|2021-01-01 00:00:00|  41|               8.3|             1|
|2021-01-01 00:00:00|  42|             28.02|             3|
|2021-01-01 00:00:00|  43|               6.8|             1|
|2021-01-01 00:00:00|  47|             63.19|             2|
|2021-01-01 00:00:00|  55|             57.25|             1|
|2021-01-01 00:00:00|  61|43.760000000000005|             2|
|2021-01-01 00:00:00|  63|             41.99|             1|
|2021-01-01 00:00:00|  74|             86.57|             7|
|2021-01-01 00:00:00|  75|             34.36|             3|
|2021-01-01 00:00:00|  7

In [19]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

In [13]:
from pathlib import Path

paths = [str(p) for p in Path("data/pq/yellow/2021").iterdir() if p.is_dir()]

df_yellow = spark.read.parquet(*paths)

In [14]:
df_yellow.createOrReplaceTempView('yellow')

In [15]:
df_yellow_revenue = spark.sql("""
SELECT 
 
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records

FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2021-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [17]:
df_yellow_revenue.show()

[Stage 23:=========================================>                (5 + 2) / 7]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2021-01-01 00:00:00|   4|              64.4|             3|
|2021-01-01 00:00:00|   7|              71.3|             2|
|2021-01-01 00:00:00|  10|              52.4|             1|
|2021-01-01 00:00:00|  13|              73.9|             3|
|2021-01-01 00:00:00|  17|              32.3|             2|
|2021-01-01 00:00:00|  24|            120.54|             7|
|2021-01-01 00:00:00|  25|             19.04|             1|
|2021-01-01 00:00:00|  28|             44.16|             1|
|2021-01-01 00:00:00|  32|             30.32|             1|
|2021-01-01 00:00:00|  41| 94.78999999999999|             5|
|2021-01-01 00:00:00|  42|62.099999999999994|             2|
|2021-01-01 00:00:00|  43|            352.23|            22|
|2021-01-01 00:00:00|  45|             93.78|             4|
|2021-01-01 00:00:00|  4

In [20]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [23]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [24]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [26]:
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

In [27]:
df_join.show()

[Stage 69:=========================================>                (5 + 2) / 7]

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2021-01-01 00:00:00|   7|             61.47|                   2|              71.3|                    2|
|2021-01-01 00:00:00|  35|              50.3|                   2|              NULL|                 NULL|
|2021-01-01 00:00:00|  39|              36.0|                   1|              NULL|                 NULL|
|2021-01-01 00:00:00|  42|             28.02|                   3|62.099999999999994|                    2|
|2021-01-01 00:00:00|  61|43.760000000000005|                   2|            106.06|                    4|
|2021-01-01 00:00:00|  72|              NULL|                NULL|              15.0|                    1|
|2021-01-01 00:00:00|  88|  

In [28]:
df_join.write.parquet('data/report/revenue/total')

In [29]:
df_join = spark.read.parquet('data/report/revenue/total')

In [32]:
df_zones = spark.read.parquet('zones/')

In [34]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [36]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

In [35]:
df_result.show()

+-------------------+----+------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2021-01-01 00:00:00|   4|        NULL|                NULL|              64.4|                    3|         4|Manhattan|       Alphabet City| Yellow Zone|
|2021-01-01 00:00:00|  17|      102.34|                   3|              32.3|                    2|        17| Brooklyn|             Bedford|   Boro Zone|
|2021-01-01 00:00:00|  41|         8.3|                   1| 94.78999999999999|                    5|        41|Manhattan|      Central Harlem|   Boro Zone|
|2021-01-01 00:00:00|  49|        NULL|                NUL

In [30]:
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

In [31]:
df_join.show()

+-------------------+----+------------+--------------------+------------------+---------------------+
|               hour|zone|green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------+--------------------+------------------+---------------------+
|2021-01-01 00:00:00|   4|        NULL|                NULL|              64.4|                    3|
|2021-01-01 00:00:00|  17|      102.34|                   3|              32.3|                    2|
|2021-01-01 00:00:00|  41|         8.3|                   1| 94.78999999999999|                    5|
|2021-01-01 00:00:00|  49|        NULL|                NULL|247.57000000000002|                    6|
|2021-01-01 00:00:00|  51|        NULL|                NULL|             17.18|                    1|
|2021-01-01 00:00:00|  77|        NULL|                NULL|             37.06|                    1|
|2021-01-01 00:00:00|  79|        NULL|                NULL|428.36000000000007|   